<div style="text-align: right;">
  <img src="https://raw.githubusercontent.com/exasol/ai-lab/refs/heads/main/assets/Exasol_Logo_2025_Dark.svg" style="width:200px; margin: 10px;" />
</div>

# Working with Exasol using JupySQL

In this notebook, we will demonstrate how to use JupySQL to query and analyze Exasol data directly in Jupyter notebooks. JupySQL provides a seamless way to write SQL in notebook cells while maintaining the full power of Python for visualization and analysis.

### What is JupySQL?

JupySQL is a modern SQL client for Jupyter notebooks that enables you to:
- Write SQL directly in notebook cells using magic commands (`%sql` and `%%sql`)
- Avoid verbose Python connection code and cursor management
- Get beautifully formatted query results automatically
- Seamlessly convert results to pandas DataFrames for analysis
- Bridge SQL analytics with Python's visualization and ML libraries

### Why use JupySQL with Exasol?

For data analysts and engineers working with Exasol, JupySQL offers several advantages:

* **Better developer experience** - Prototype queries interactively, see results immediately, and iterate quickly without context switching between SQL and Python.
* **Visualization ready** - Query Exasol data and immediately visualize it with matplotlib, plotly, or other Python libraries.
* **Reproducible analysis** - Your SQL queries, results, and visualizations live together in one shareable notebook, perfect for documentation and collaboration.
* **Production-ready connections** - Built on SQLAlchemy, you get reliable connection pooling and parameterized queries.

The notebook is organized as a quickstart tutorial in which we will be looking at US flight delays. In particular, we will explore the delay caused by the carrier. We will rank the carriers using the delay as the performance metric. The data is publicly accessible at the [Bureau of Transportation Statistics](https://www.transtats.bts.gov/Homepage.asp) of the US Department of Transportation.

### Prerequisites

Prior to using this notebook, the following steps need to be completed:
1. [Configure the AI-Lab](../main_config.ipynb).
2. [Load the US Flights data](../data/data_flights.ipynb).

### How to Use This Notebook:
- If you're running in the Exasol AI-Lab environment, then you can use the AI-Lab configuration cells (labeled as "Option A" in the **Configuration** section).
- If you're running in a standard Jupyter environment (VS Code, JupyterLab, etc.), then please provide your own Exasol connection details (labeled as "Option B" in the **Configuration** section).
- Make sure to run cells sequentially, especially the connection setup cells before running any queries.

**Note:** The AI-Lab specific cells (using `%run utils/access_store_ui.ipynb` and `%run ../utils/jupysql_init.ipynb`) can be ignored if you're not running in the AI-Lab environment.

## Configuration

Choose the configuration approach that matches your environment:

- **Option A**: For Exasol AI-Lab users (skip to next section)
- **Option B**: For portable use in any Jupyter environment (VS Code, JupyterLab, Google Colab, etc.)

---

### Option A: Configuration for Exasol AI-Lab

**Use this section ONLY if you're running in the Exasol AI-Lab environment.**

The AI-Lab provides pre-configured connection settings that handle all setup automatically.

In [ ]:
# Open the secure configuration storage (AI-Lab only)
from exasol.nb_connector.ui.access import access_store
display(access_store.get_access_store('../'))

In [ ]:
# Load JupySQL and connect using AI-Lab configuration (AI-Lab only)
from exasol.nb_connector.ui.common import jupysql
jupysql.init(ai_lab_config)

---

**AI-Lab users:** Skip to the "Explore the Data" section below. The connection is now ready!

---

### Option B: Portable Configuration (Any Jupyter Environment)

<span style="color:red">**Use this section if you're NOT using Exasol AI-Lab.** This works in VS Code, JupyterLab, Google Colab, or any standard Jupyter environment.

**Prerequisites:** Make sure you have installed the required packages:

In [ ]:
pip install jupysql sqlalchemy-exasol

The **sqlalchemy-exasol** package is the dedicated SQLAlchemy dialect for connecting Python to an Exasol database. It translates standard SQL commands into the specific syntax and protocols used by **Exasol**. For more information, check out its [User Guide](https://exasol.github.io/sqlalchemy-exasol/master/user_guide/index.html#user-guide).




#### Step 1: Load the JupySQL Extension

First, load the SQL magic extension (one-time setup per notebook session):

In [ ]:
# Load the SQL magic extension
%load_ext sql

#### Step 2: Connect to Your Exasol Database

**Option A: Use Environmental Variables for Connection Values**

For your OS, figure out how to set environment variables. For the code below, you will need to set:
- EXASOL_USER
- EXASOL_PASSWORD
- EXASOL_HOST
- EXASOL_PORT (if not 8563)
- EXASOL_SCHEMA (if not PUBLIC)

In [ ]:
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
from os import environ

schema = environ.get("EXASOL_SCHEMA")
ssl_certificate = environ.get("EXASOL_SSL_CERTIFICATE")


url_object = URL.create(
    drivername="exa+websocket",
    username=environ.get("EXASOL_USER"),
    password=environ.get("EXASOL_PASSWORD"),
    host=environ.get("EXASOL_HOST"),
    port=environ.get("EXASOL_PORT"),
    query={
        "SSLCertificate": ssl_certificate,
    },
)
engine = create_engine(url_object)

%sql engine
%config SqlMagic.short_errors = False
%sql OPEN SCHEMA {{schema}}

#### Alternatives

JupySQL can automatically connect using the `DATABASE_URL` environment variable:

```python
from os import environ
environ["DATABASE_URL"] = f"exa+websocket://{EXASOL_USER}:{EXASOL_PASSWORD}@{EXASOL_HOST}:{EXASOL_PORT}/{EXASOL_SCHEMA}"
```

Then simply run:
```python
%sql
```

**For more connection options and security best practices**, see the [JupySQL Connection Documentation](https://jupysql.readthedocs.io/en/latest/connecting.html).

---

**Connection established!** You're now ready to query Exasol using `%%sql` magic commands.

---

## Analyze your Data

Let's start by looking at the flight delay data. Notice how we can write SQL directly in the cell with the `%%sql` magic command - no Python string wrapping needed!

In [ ]:
%%sql
SELECT *
FROM US_FLIGHTS
LIMIT 5

### Filter Data

Should we compute statistics on all records? What about canceled or diverted flights? Let's examine these edge cases.

First, let's look at canceled flights:

In [ ]:
%%sql
SELECT *
FROM US_FLIGHTS
WHERE CANCELLED = TRUE
LIMIT 5

And diverted flights:

In [ ]:
%%sql
SELECT *
FROM US_FLIGHTS
WHERE DIVERTED = TRUE
LIMIT 5

There's no delay information for canceled or diverted flights, so we'll exclude them from our analysis.

### Aggregate Statistics by Carrier

Let's compute delay statistics for each carrier, excluding canceled and diverted flights:

In [ ]:
%%sql
SELECT 
    OP_CARRIER_AIRLINE_ID,
    SUM(CARRIER_DELAY) AS combined_delay,
    COUNT(CARRIER_DELAY) AS total_delayed,
    COUNT(*) AS total_flights
FROM US_FLIGHTS
WHERE CANCELLED = FALSE 
    AND DIVERTED = FALSE
GROUP BY OP_CARRIER_AIRLINE_ID
LIMIT 5

### Calculate Performance Metrics

Now let's add calculated columns for the percentage of flights delayed and average delay per flight:

In [ ]:
%%sql
SELECT 
    OP_CARRIER_AIRLINE_ID,
    SUM(CARRIER_DELAY) AS combined_delay,
    COUNT(CARRIER_DELAY) AS total_delayed,
    COUNT(*) AS total_flights,
    ROUND(100.0 * COUNT(CARRIER_DELAY) / COUNT(*), 2) AS percent_delayed,
    ROUND(SUM(CARRIER_DELAY) / COUNT(*), 2) AS delay_per_flight
FROM US_FLIGHTS
WHERE CANCELLED = FALSE 
    AND DIVERTED = FALSE
GROUP BY OP_CARRIER_AIRLINE_ID
LIMIT 5

### Join with Airline Names

Let's make our results more readable by joining with the airline names table:

In [ ]:
%%sql
SELECT 
    a.CARRIER_NAME,
    f.OP_CARRIER_AIRLINE_ID,
    SUM(f.CARRIER_DELAY) AS combined_delay,
    COUNT(f.CARRIER_DELAY) AS total_delayed,
    COUNT(*) AS total_flights,
    ROUND(100.0 * COUNT(f.CARRIER_DELAY) / COUNT(*), 2) AS percent_delayed,
    ROUND(SUM(f.CARRIER_DELAY) / COUNT(*), 2) AS delay_per_flight
FROM US_FLIGHTS f
INNER JOIN US_AIRLINES a 
    ON f.OP_CARRIER_AIRLINE_ID = a.OP_CARRIER_AIRLINE_ID
WHERE f.CANCELLED = FALSE 
    AND f.DIVERTED = FALSE
GROUP BY a.CARRIER_NAME, f.OP_CARRIER_AIRLINE_ID
LIMIT 5

### Rank Airlines by Performance

Finally, let's order the results to see the worst-performing airlines first (highest average delay per flight):

In [ ]:
%%sql
SELECT 
    a.CARRIER_NAME,
    ROUND(100.0 * COUNT(f.CARRIER_DELAY) / COUNT(*), 2) AS percent_delayed,
    ROUND(SUM(f.CARRIER_DELAY) / COUNT(*), 2) AS delay_per_flight
FROM US_FLIGHTS f
INNER JOIN US_AIRLINES a 
    ON f.OP_CARRIER_AIRLINE_ID = a.OP_CARRIER_AIRLINE_ID
WHERE f.CANCELLED = FALSE 
    AND f.DIVERTED = FALSE
GROUP BY a.CARRIER_NAME, f.OP_CARRIER_AIRLINE_ID
ORDER BY delay_per_flight DESC
LIMIT 10

### Analyzing Delay Patterns by State

Let's explore which origin states have the most delays:

In [ ]:
%%sql
SELECT 
    ORIGIN_STATE_ABR AS state_abr,
    COUNT(*) AS total_flights,
    ROUND(AVG(DEP_DELAY), 2) AS avg_departure_delay,
    ROUND(AVG(ARR_DELAY), 2) AS avg_arrival_delay
FROM US_FLIGHTS
WHERE CANCELLED = FALSE 
    AND DIVERTED = FALSE
    AND DEP_DELAY IS NOT NULL
GROUP BY ORIGIN_STATE_ABR
HAVING COUNT(*) >= 10000
ORDER BY avg_departure_delay DESC
LIMIT 10

## Combining Python and JupySQL

One of JupySQL's strengths is seamless integration with Python. Let's store query results and create a visualization.

We can save results to a variable and convert to pandas:

In [ ]:
%%sql result <<
SELECT
    a.carrier_name,
    ROUND(100.0 * COUNT(f.carrier_delay) / COUNT(*), 2) AS percent_delayed,
    ROUND(SUM(f.carrier_delay) / COUNT(*), 2) AS delay_per_flight
FROM us_flights f
INNER JOIN us_airlines a
    ON f.op_carrier_airline_id = a.op_carrier_airline_id
WHERE f.cancelled = false
  AND f.diverted = false
GROUP BY a.carrier_name, f.op_carrier_airline_id
ORDER BY delay_per_flight DESC
LIMIT 10

In [ ]:
# Convert to pandas DataFrame
df = result.DataFrame()
df

### Creating visualizations

Now, we can use matplotlib or any other Python visualization library:

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.barh(df['carrier_name'], df['delay_per_flight'])
plt.xlabel('Average Delay per Flight (minutes)')
plt.ylabel('Carrier')
plt.title('Top 10 Airlines by Average Delay per Flight')
plt.gca().invert_yaxis()  # Highest delay at top
plt.tight_layout()
plt.show()

### Using Parameterized Queries

JupySQL supports parameterized queries, making it easy to build dynamic analyses. First, enable named parameters:

In [ ]:
# Enable named parameters feature in JupySQL
%config SqlMagic.named_parameters="enabled"

In [ ]:
# Set a Python variable
min_flights = 50000

In [ ]:
%%sql
SELECT
    a.CARRIER_NAME,
    COUNT(*) AS flight_count,
    ROUND(SUM(f.CARRIER_DELAY) / COUNT(*), 2) AS delay_per_flight
FROM US_FLIGHTS f
INNER JOIN US_AIRLINES a
    ON f.OP_CARRIER_AIRLINE_ID = a.OP_CARRIER_AIRLINE_ID
WHERE f.CANCELLED = FALSE
  AND f.DIVERTED = FALSE
GROUP BY a.CARRIER_NAME, f.OP_CARRIER_AIRLINE_ID
HAVING COUNT(*) >= :min_flights
ORDER BY delay_per_flight DESC

## Understanding JupySQL and SQLAlchemy

Behind the scenes, JupySQL uses **SQLAlchemy**, Python's most popular SQL toolkit and Object Relational Mapper (ORM). SQLAlchemy provides the robust database connectivity layer that powers JupySQL's magic commands.

You can also use SQLAlchemy directly when you need more programmatic control over your database operations:

In [ ]:
from exasol.nb_connector.connections import open_sqlalchemy_connection
from sqlalchemy import text

# Get a SQLAlchemy engine using AI-Lab config
engine = open_sqlalchemy_connection(ai_lab_config)

# Execute a query using SQLAlchemy directly
query = text("""
    SELECT CARRIER_NAME, 
           COUNT(*) as total_flights
    FROM US_FLIGHTS f
    JOIN US_AIRLINES a ON a.OP_CARRIER_AIRLINE_ID = f.OP_CARRIER_AIRLINE_ID
    WHERE NOT (CANCELLED OR DIVERTED)
    GROUP BY CARRIER_NAME
    ORDER BY total_flights DESC
    LIMIT 5
""")

with engine.connect() as conn:
    result = conn.execute(query)
    for row in result:
        print(f"{row[0]}: {row[1]:,} flights")

**When to use SQLAlchemy directly:**
- Building complex application logic with transactions
- Using SQLAlchemy's ORM features for object-relational mapping
- Needing fine-grained control over connection pooling
- Programmatically constructing complex queries

**When to use JupySQL:**
- Interactive data exploration and analysis
- Quick prototyping of SQL queries
- Creating data narratives in notebooks
- Teaching and demonstrating SQL concepts

Both approaches work seamlessly with Exasol, and you can even use them together in the same notebook!

## Summary

In this notebook, we've demonstrated how JupySQL provides a natural way to work with Exasol data in Jupyter notebooks:

1. **Simple setup** - One line to load the extension, one line to connect
2. **Clean SQL syntax** - Write SQL directly without Python string wrapping
3. **Immediate results** - Beautifully formatted output automatically
4. **Python integration** - Easy conversion to pandas for visualization and further analysis
5. **Parameterized queries** - Dynamic queries using Python variables

JupySQL bridges the gap between SQL analytics and Python's rich ecosystem, making it ideal for:
- Data exploration and ad-hoc analysis
- Creating reproducible analytical reports
- Prototyping SQL queries before moving them to production
- Teaching and demonstrating SQL concepts
- Building data-driven narratives that combine queries, results, and visualizations

For more information on JupySQL, visit the [official documentation](https://jupysql.readthedocs.io/en/latest/).